# Базовая оптимизация параметров SIR
Эволюционный поиск параметров, уменьшающих пик и смертность.

In [1]:
using DrWatson
@quickactivate "project"

using BlackBoxOptim
using CSV
using DataFrames

include(srcdir("sir_model.jl"))

script_name = splitext(basename(PROGRAM_FILE))[1]
mkpath(datadir(script_name))

const OPTIMIZATION_SEEDS = (101, 202, 303)

function objective(x)
    beta_und = x[1]
    detection_time = round(Int, x[2])
    death_rate = x[3]
    score = 0.0
    for seed in OPTIMIZATION_SEEDS
        model = initialize_sir(
            Ns = [1000, 1000, 1000],
            Is = [1, 0, 0],
            beta_und = beta_und,
            beta_det = beta_und / 10,
            infection_period = 14,
            detection_time = clamp(detection_time, 2, 12),
            death_rate = clamp(death_rate, 0.0, 0.1),
            reinfection_probability = 0.1,
            migration_intensity = 0.2,
            seed = seed,
        )
        df = simulate_sir!(model, 60)
        metrics = summarize_dynamics(df, 3000)
        score += metrics.death_fraction + 0.7 * metrics.peak
    end
    return score / length(OPTIMIZATION_SEEDS)
end

res = bboptimize(objective;
    SearchRange = [(0.25, 0.8), (2.0, 12.0), (0.01, 0.08)],
    NumDimensions = 3,
    MaxSteps = 25,
    TraceMode = :verbose,
)

best = best_candidate(res)
beta_und = best[1]
detection_time = round(Int, best[2])
death_rate = best[3]
objective_value = best_fitness(res)

out = DataFrame(
    beta_und = [beta_und],
    beta_det = [beta_und / 10],
    detection_time = [detection_time],
    death_rate = [death_rate],
    objective = [objective_value],
)
CSV.write(datadir(script_name, "optimization_basic.csv"), out)
println(out)
println("saved: ", datadir(script_name, "optimization_basic.csv"))

Starting optimization with optimizer DiffEvoOpt{FitPopulation{Float64}, RadiusLimitedSelector, BlackBoxOptim.AdaptiveDiffEvoRandBin{3}, RandomBound{ContinuousRectSearchSpace}}
0.00 secs, 0 evals, 0 steps


DE modify state:
0.60 secs, 2 evals, 1 steps

, fitness=0.035877778
DE modify state:
1.13 secs, 4 evals, 2 steps

, improv/step: 0.500 (last = 1.0000), fitness=0.035877778
DE modify state:
2.07 secs, 8 evals, 4 steps, improv/step: 0.250 (last = 0.0000)

, fitness=0.001088889
DE modify state:
3.02 secs, 12 evals, 6 steps

...

, improv/step: 0.455 (last = 0.5000), fitness=0.000544444
DE modify state:
11.62 secs, 46 evals, 24 steps, improv/step: 0.417 (last = 0.0000)

, fitness=0.000544444
DE modify state:

Optimization stopped after 26 steps and 12.58 seconds


Termination reason: Max number of steps (25) reached
Steps per second = 2.07
Function evals per second = 3.97
Improvements/step = 0.44000
Total function evaluations = 50


Best candidate found: 

[0.272991, 2.04809, 0.0235555]

Fitness: 0.000544444

1×5 DataFrame
 Row │ beta_und  beta_det   detection_time  death_rate  objective   
     │ Float64   Float64    Int64           Float64     Float64     
─────┼──────────────────────────────────────────────────────────────
   1 │ 0.272991  0.0272991               2   0.0235555  0.000544444


saved: /home/lilidji/github-push-work/labs/lab04/project/data/optimization_basic.csv
